In [4]:
import json

pedido_app = {
    "id_pedido": 5001,
    "cliente": {
        "id_cliente": 105,
        "nombre": "Carlos",
        "email": "carlos@email.com"
    },
    "items": [
        {"producto": "Pizza", "cantidad": 2, "precio_unitario": 150.0},
        {"producto": "Refresco", "cantidad": 1, "precio_unitario": 35.0}
    ],
    "direccion_entrega": {
        "calle": "Av. Vallarta",
        "numero": "1234",
        "cp": "45000"
    }
}

print(json.dumps(pedido_app, indent=2))

{
  "id_pedido": 5001,
  "cliente": {
    "id_cliente": 105,
    "nombre": "Carlos",
    "email": "carlos@email.com"
  },
  "items": [
    {
      "producto": "Pizza",
      "cantidad": 2,
      "precio_unitario": 150.0
    },
    {
      "producto": "Refresco",
      "cantidad": 1,
      "precio_unitario": 35.0
    }
  ],
  "direccion_entrega": {
    "calle": "Av. Vallarta",
    "numero": "1234",
    "cp": "45000"
  }
}


In [5]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript('''
CREATE TABLE Clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT,
    email TEXT
);

CREATE TABLE Direcciones (
    id_direccion INTEGER PRIMARY KEY AUTOINCREMENT,
    calle TEXT,
    numero TEXT,
    cp TEXT,
    id_cliente INTEGER
);

CREATE TABLE Pedidos (
    id_pedido INTEGER PRIMARY KEY,
    id_cliente INTEGER,
    id_direccion INTEGER
);

CREATE TABLE Items_Pedido (
    id_item INTEGER PRIMARY KEY AUTOINCREMENT,
    id_pedido INTEGER,
    producto TEXT,
    cantidad INTEGER,
    precio_unitario REAL
);
''')

In [6]:
# Cliente
cliente = pedido_app["cliente"]
cursor.execute(
    "INSERT INTO Clientes VALUES (?, ?, ?)",
    (cliente["id_cliente"], cliente["nombre"], cliente["email"])
)

# Dirección
direccion = pedido_app["direccion_entrega"]
cursor.execute(
    "INSERT INTO Direcciones (calle, numero, cp, id_cliente) VALUES (?, ?, ?, ?)",
    (direccion["calle"], direccion["numero"], direccion["cp"], cliente["id_cliente"])
)

id_direccion = cursor.lastrowid

# Pedido
cursor.execute(
    "INSERT INTO Pedidos VALUES (?, ?, ?)",
    (pedido_app["id_pedido"], cliente["id_cliente"], id_direccion)
)

# Items
for item in pedido_app["items"]:
    cursor.execute(
        "INSERT INTO Items_Pedido (id_pedido, producto, cantidad, precio_unitario) VALUES (?, ?, ?, ?)",
        (pedido_app["id_pedido"], item["producto"], item["cantidad"], item["precio_unitario"])
    )

conn.commit()

print("✔ Inserción completada")

✔ Inserción completada


In [7]:
cursor.execute("SELECT * FROM Clientes")
print(cursor.fetchall())

cursor.execute("SELECT * FROM Items_Pedido")
print(cursor.fetchall())

[(105, 'Carlos', 'carlos@email.com')]
[(1, 5001, 'Pizza', 2, 150.0), (2, 5001, 'Refresco', 1, 35.0)]


In [8]:
query = """
SELECT 
    p.id_pedido,
    c.nombre,
    c.email,
    d.calle,
    d.numero,
    d.cp,
    i.producto,
    i.cantidad,
    i.precio_unitario
FROM Pedidos p
JOIN Clientes c ON p.id_cliente = c.id_cliente
JOIN Direcciones d ON p.id_direccion = d.id_direccion
JOIN Items_Pedido i ON p.id_pedido = i.id_pedido
WHERE p.id_pedido = 5001;
"""

cursor.execute(query)

for fila in cursor.fetchall():
    print(fila)

(5001, 'Carlos', 'carlos@email.com', 'Av. Vallarta', '1234', '45000', 'Pizza', 2, 150.0)
(5001, 'Carlos', 'carlos@email.com', 'Av. Vallarta', '1234', '45000', 'Refresco', 1, 35.0)


In [9]:
import time
import random

tabla_sql = list(range(1, 50000, 2))

print("Simulando SQL...")
inicio = time.time()

for _ in range(10000):
    nuevo_id = random.randint(1, 50000)
    for i, val in enumerate(tabla_sql):
        if val > nuevo_id:
            tabla_sql.insert(i, nuevo_id)
            break
    else:
        tabla_sql.append(nuevo_id)

fin = time.time()
tiempo_sql = fin - inicio

print("Tiempo SQL:", tiempo_sql)

Simulando SQL...
Tiempo SQL: 24.922845125198364


In [10]:
data_lake = list(range(1, 50000, 2))

print("Simulando Big Data...")
inicio = time.time()

nuevos = [random.randint(1, 50000) for _ in range(10000)]

for n in nuevos:
    data_lake.append(n)

fin = time.time()
tiempo_bigdata = fin - inicio

print("Tiempo Big Data:", tiempo_bigdata)
print("Aceleración:", tiempo_sql / tiempo_bigdata)

Simulando Big Data...
Tiempo Big Data: 0.10830903053283691
Aceleración: 230.10865301432372


In [ ]:
class NodoBD:
    def __init__(self, nombre):
        self.nombre = nombre
        self.datos = {"saldo_cliente_1": 1000}
        self.conectado = True

    def leer_saldo(self):
        return self.datos["saldo_cliente_1"]

    def actualizar_saldo(self, monto):
        self.datos["saldo_cliente_1"] = monto


nodo_a = NodoBD("Guadalajara")
nodo_b = NodoBD("Monterrey")

# Actualización
nodo_a.actualizar_saldo(800)
print(nodo_a.leer_saldo())

# Simular caída
nodo_b.conectado = False

if nodo_b.conectado:
    nodo_b.actualizar_saldo(800)
else:
    print("Nodo B sin conexión")

In [12]:
# CP (correcto para banco)
if not nodo_b.conectado:
    print("ERROR: Servicio no disponible (consistencia)")
else:
    print("Saldo:", nodo_b.leer_saldo())

ERROR: Servicio no disponible (consistencia)


## Impedance Mismatch
 Explique con sus propias palabras qué es el Impedance Mismatch. ¿Por qué cree que los desarrolladores prefieren usar bases de datos Documentales (como MongoDB) para aplicaciones modernas basadas en JSON?
Es el problema entre objetos (JSON) y tablas SQL.
SQL obliga a dividir datos complejos en varias tablas.
## JOINs
Analice el resultado de su consulta SQL con JOINs del Ejercicio 1.3. ¿Por qué se repiten los datos del cliente (nombre, email) por cada artículo que compró? ¿Es esto eficiente para transmitirlo por una red hacia una aplicación móvil?
Los datos se repiten porque hay múltiples filas por producto.
Esto genera redundancia y no es eficiente.
## Big Data
El enfoque "Append-Only" usado en Big Data es rapidísimo para guardar, pero dificulta la lectura porque los datos están desordenados. ¿Qué estrategia arquitectónica usa el ecosistema Hadoop/Spark para poder leer estos datos masivos rápidamente a pesar del desorden? (Pista: Involucra la palabra "Distribuido").
Usa procesamiento distribuido para leer datos en paralelo.
## CAP
¿Qué decisión tomó frente al Teorema CAP (CP o AP) y por qué esa elección es la correcta para un sistema bancario?
Elegí CP porque en un banco es más importante la consistencia.
Es mejor dar error que mostrar datos incorrectos.